In [ ]:
# Use q to exit, and a and d for previous or next filter (first filter doesnt show when exiting recording)
# sue 'a' and 'd' to navigate between filters
# during the recording, use windows snipping tool for some static objects and save them in "temmplates" folder (level 6)

In [1]:
import cv2
import numpy as np
import torch
import time
from ultralytics import YOLO
import os
from ultralytics import YOLOWorld
from collections import defaultdict


torch.cuda.is_available()

# Define a color palette with distinct colors
COLOR_PALETTE = [
    (255, 0, 0),     # Blue
    (0, 255, 0),     # Green
    (0, 0, 255),     # Red
    (255, 255, 0),   # Cyan
    (255, 0, 255),   # Magenta
    (0, 255, 255),   # Yellow
    (128, 0, 0),     # Maroon
    (0, 128, 0),     # Dark Green
    (0, 0, 128),     # Navy
    (128, 128, 0)    # Olive
]

def initialize_model():

    try:
        model = YOLOWorld("yolov8s-worldv2.pt")

        if torch.cuda.is_available():
            model = model.to('cuda')
            print(f"Using GPU ({torch.cuda.get_device_name(0)}) for inference.")
        else:
            print("GPU not available, using CPU for inference.")
            print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
            print(f"torch.version.cuda: {torch.version.cuda}")
            print(f"torch.version: {torch.__version__}")
            print("If you have a GPU, ensure PyTorch is installed with CUDA support.")
            print("Visit https://pytorch.org/get-started/locally/ for installation instructions.")
    except Exception as e:
        print(f"Error loading model: {e}")
        exit()
    return model

def initialize_camera():
    # Try different camera indices (0, 1, 2, etc.) to find an available camera
    for i in range(5):  # Adjust the range if you have more potential cameras
        cap = cv2.VideoCapture(i)
        if cap.isOpened():
            print(f"Camera {i} opened successfully.")
            break
        else:
            print(f"Camera {i} not found.")

    if not cap.isOpened():
        print("Error: Could not open any video capture devices.")
        cap.release()
        cv2.destroyAllWindows()
        exit()

    # Set desired frame size to 1920x1200
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1200)
    frame_width = 1280
    frame_height = 1080
    # Get frame dimensions
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if frame_width != 1920 or frame_height != 1200:
        print(f"Desired resolution 1920x1200 not supported. Using {frame_width}x{frame_height} instead.")
    else:
        print(f"Camera initialized with resolution: {frame_width}x{frame_height}")
    return cap, frame_width, frame_height

def visualize_level_1(frame, block_size=25):
    """
    Visualizes Level 1: Left side pixelated, right side with RGB values over black background.
    """
    height, width = frame.shape[:2]

    # Pixelate the entire frame
    pixelated_frame = pixelate_image(frame, block_size)

    # Define the right half region
    x_start = width // 2
    y_start = 0
    region_width = width // 2
    region_height = height

    # Replace the right half with RGB values over black background
    regions = [(x_start, y_start, region_width, region_height)]
    frame_with_numbers, explanation_numbers = visualize_rgb_region(frame, regions, block_size)

    # Combine the pixelated left half and the right half with numbers
    combined_img = pixelated_frame.copy()
    combined_img[:, x_start:] = frame_with_numbers[:, x_start:]

    explanation = "Left: Pixelated image. Right: RGB values over black background. Can you still recognice yourself in the numbers?"
    return combined_img, explanation

def pixelate_image(frame, block_size=25):
    """
    Pixelates the entire image.
    """
    height, width = frame.shape[:2]
    temp = cv2.resize(frame, (width // block_size, height // block_size), interpolation=cv2.INTER_LINEAR)
    pixelated_frame = cv2.resize(temp, (width, height), interpolation=cv2.INTER_NEAREST)
    return pixelated_frame

def visualize_rgb_region(frame, regions, block_size=25):
    """
    Visualizes the RGB values for specific regions of a frame.
    """
    frame_with_rgb_numbers = frame.copy()
    for region in regions:
        x_start, y_start, width, height = region
        # Ensure the region is within the frame boundaries
        x_end = min(x_start + width, frame.shape[1])
        y_end = min(y_start + height, frame.shape[0])
        roi = frame[y_start:y_end, x_start:x_end].copy()
        num_blocks_x = width // block_size
        num_blocks_y = height // block_size

        # Create a black background for the region
        number_img = np.zeros_like(roi)

        # Calculate font and padding based on block size
        font_scale = block_size / 80.0
        text_y_offset = max(2, int(block_size / 4))
        padding_x = max(2, int(block_size / 4))
        padding_y = max(2, int(block_size / 4))

        for y in range(num_blocks_y):
            for x in range(num_blocks_x):
                block_x_start = x * block_size
                block_y_start = y * block_size
                block_x_end = min(block_x_start + block_size, roi.shape[1])
                block_y_end = min(block_y_start + block_size, roi.shape[0])

                # Get the block
                block = roi[block_y_start:block_y_end, block_x_start:block_x_end]

                # Calculate the mean color for the block
                mean_color_bgr = block.mean(axis=(0, 1)).astype(int)

                # Prepare RGB text values
                r_val = f"{mean_color_bgr[2]:03d}"  # Red channel
                g_val = f"{mean_color_bgr[1]:03d}"  # Green channel
                b_val = f"{mean_color_bgr[0]:03d}"  # Blue channel

                # Text color is the mean color of the block
                text_color = tuple(map(int, mean_color_bgr))

                # Draw the text on the black background
                text_x = block_x_start + padding_x
                text_y = block_y_start + padding_y
                cv2.putText(number_img, f"[{r_val}", (text_x, text_y),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)
                cv2.putText(number_img, f" {g_val}", (text_x, text_y + text_y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)
                cv2.putText(number_img, f" {b_val}]", (text_x, text_y + 2 * text_y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)

        # Replace the region in the frame with the number_img
        frame_with_rgb_numbers[y_start:y_end, x_start:x_end] = number_img

    explanation = "A filter takes all the surrounding RGB values of each pixel to compute a new value. This filter increases the saturation."
    return frame_with_rgb_numbers, explanation

def visualize_rgb_region_black_background(frame, regions, block_size=25):
    """
    Replaces specified regions with black background and overlays RGB values.
    """
    frame_with_rgb_numbers = frame.copy()
    for region in regions:
        x_start, y_start, width, height = region
        # Ensure the region is within the frame boundaries
        x_end = min(x_start + width, frame.shape[1])
        y_end = min(y_start + height, frame.shape[0])
        # Replace the region with black background
        frame_with_rgb_numbers[y_start:y_end, x_start:x_end] = 0
        roi = frame[y_start:y_end, x_start:x_end]
        num_blocks_x = width // block_size
        num_blocks_y = height // block_size

        # Calculate font and padding based on block size
        font_scale = block_size / 80.0
        text_y_offset = max(2, int(block_size / 4))
        padding_x = max(2, int(block_size / 4))
        padding_y = max(2, int(block_size / 4))

        for y_idx in range(num_blocks_y):
            for x_idx in range(num_blocks_x):
                block_x_start = x_idx * block_size
                block_y_start = y_idx * block_size
                block_x_end = min(block_x_start + block_size, roi.shape[1])
                block_y_end = min(block_y_start + block_size, roi.shape[0])

                # Get the block
                block = roi[block_y_start:block_y_end, block_x_start:block_x_end]

                # Calculate the mean color for the block
                mean_color_bgr = block.mean(axis=(0, 1)).astype(int)

                # Prepare RGB text values
                r_val = f"{mean_color_bgr[2]:03d}"  # Red channel
                g_val = f"{mean_color_bgr[1]:03d}"  # Green channel
                b_val = f"{mean_color_bgr[0]:03d}"  # Blue channel

                # Text color is the mean color of the block
                text_color = tuple(map(int, mean_color_bgr))

                # Draw the text on the black background
                text_x = x_start + block_x_start + padding_x
                text_y = y_start + block_y_start + padding_y

                # Draw the text
                cv2.putText(frame_with_rgb_numbers, f"[{r_val}", (text_x, text_y),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)
                cv2.putText(frame_with_rgb_numbers, f" {g_val}", (text_x, text_y + text_y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)
                cv2.putText(frame_with_rgb_numbers, f" {b_val}]", (text_x, text_y + 2 * text_y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)

    explanation = "A filter takes all the surrounding RGB values of each pixel to compute a new value. This filter increases the saturation."
    return frame_with_rgb_numbers, explanation

def enhance_saturation(frame, saturation_threshold=100, saturation_factor=1.5):
    """
    Enhances high-saturation colors and converts low/normal saturation areas to grayscale.
    """
    # Convert BGR to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(float)

    # Split into channels
    h, s, v = cv2.split(hsv)

    # Create mask for high saturation
    high_sat_mask = s > saturation_threshold

    # Enhance saturation
    s[high_sat_mask] *= saturation_factor
    s[s > 255] = 255  # Cap saturation at 255

    # Convert low/normal saturation areas to grayscale by setting saturation to 0
    s[~high_sat_mask] = 0

    # Merge channels back
    hsv_enhanced = cv2.merge([h, s, v]).astype(np.uint8)

    # Convert back to BGR
    bgr_enhanced = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2BGR)

    return bgr_enhanced

def visualize_level_2(frame, filtered_frame, processed_mask, kernel_position, kernel_size=500, block_size=25):
    """
    Visualizes how convolutions work by moving a kernel over the frame.
    """
    height, width = frame.shape[:2]
    x, y = kernel_position
    x_end = min(x + kernel_size, width)
    y_end = min(y + kernel_size, height)
    
    # Update the processed_mask to include the center of the kernel
    third = kernel_size // 3
    center_x_start = x + third
    center_y_start = y + third
    center_x_end = x + 2 * third
    center_y_end = y + 2 * third

    # Ensure within bounds
    center_x_start = max(0, center_x_start)
    center_y_start = max(0, center_y_start)
    center_x_end = min(width, center_x_end)
    center_y_end = min(height, center_y_end)

    # Update processed_mask
    processed_mask[center_y_start:center_y_end, center_x_start:center_x_end] = 255

    # Create the output frame
    output_frame = frame.copy()
    # Apply the filtered_frame where processed_mask is 255
    output_frame[processed_mask == 255] = filtered_frame[processed_mask == 255]

    # Draw the kernel grid (2 vertical and 2 horizontal lines to form 9 squares)
    # Vertical lines
    v1 = x + third
    v2 = x + 2 * third
    # Horizontal lines
    h1 = y + third
    h2 = y + 2 * third

    # Draw vertical lines
    cv2.line(output_frame, (v1, y), (v1, y_end), (0, 255, 0), 2)
    cv2.line(output_frame, (v2, y), (v2, y_end), (0, 255, 0), 2)
    # Draw horizontal lines
    cv2.line(output_frame, (x, h1), (x_end, h1), (0, 255, 0), 2)
    cv2.line(output_frame, (x, h2), (x_end, h2), (0, 255, 0), 2)
    # Draw outer rectangle
    cv2.rectangle(output_frame, (x, y), (x_end, y_end), (0, 255, 0), 2)

    # Prepare the regions for RGB numbers (all squares except the center square)
    border_regions = []
    for i in range(3):  # rows
        for j in range(3):  # columns
            # Skip the center square
            if i == 1 and j == 1:
                continue
            square_x_start = x + j * third
            square_y_start = y + i * third
            border_regions.append((square_x_start, square_y_start, third, third))

    # Replace these regions with black background and overlay RGB values
    output_frame, _ = visualize_rgb_region_black_background(output_frame, border_regions, block_size)

    explanation = "A filter takes all the surrounding RGB values of each pixel to compute a new value. This filter increases the saturation."

    return output_frame, processed_mask, explanation

def visualize_level_3(frame):
    # Edge detection using Canny

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    edge_img = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    explanation = "Edge detection is one of the most common techniques in computer vision and help identify shapes."
    

    # Edge detection using Laplacian

    # gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # edges = cv2.Laplacian(gray, cv2.CV_64F)
    # edges = cv2.convertScaleAbs(edges)
    # edge_img = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    # explanation = "Step 5: Detecting edges using Laplacian for better object detection."

    # Edge detection using Sobel
    # gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
    # sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
    # edges = cv2.magnitude(sobelx, sobely)
    # edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX)
    # edge_img = cv2.cvtColor(edges.astype(np.uint8), cv2.COLOR_GRAY2BGR)
    # explanation = "Step 5: Detecting edges using Sobel for better object detection."


    return edge_img, explanation

def visualize_level_4(frame):
    """
    Simplifies the image by mapping each pixel to the nearest color in a predefined palette.
    Enhances colors and ensures consistency across frames to prevent flickering.
    """

    # Define an expanded fixed color palette (K=8)
    palette = np.array([
        [255, 0, 0],       # Red
        [0, 255, 0],       # Green
        [0, 0, 255],       # Blue
        [255, 255, 0],     # Yellow
        [255, 0, 255],     # Magenta
        [0, 255, 255],     # Cyan
        [255, 165, 0],     # Orange
        [128, 0, 128]      # Purple
    ], dtype=np.uint8)

    # Convert the palette to LAB color space for better color distance calculations
    lab_palette = cv2.cvtColor(palette.reshape(-1, 1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3)

    # Resize the frame for faster processing (optional)
    scale_percent = 50  # Resize to 50% of original size
    width = int(frame.shape[1] * scale_percent / 100)
    height = int(frame.shape[0] * scale_percent / 100)
    resized_frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)

    # Convert the resized frame to LAB color space
    lab_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2LAB)

    # Reshape the image to a 2D array of pixels
    pixel_values = lab_frame.reshape((-1, 3)).astype(np.float32)

    # Compute Euclidean distance between each pixel and each palette color
    distances = np.linalg.norm(pixel_values[:, np.newaxis] - lab_palette, axis=2)

    # Find the index of the nearest palette color for each pixel
    nearest_color_indices = np.argmin(distances, axis=1)

    # Map each pixel to the nearest palette color
    segmented_pixels = palette[nearest_color_indices]

    # Reshape back to the resized image shape
    segmented_image = segmented_pixels.reshape(resized_frame.shape)

    # Resize back to original size
    output = cv2.resize(segmented_image, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)

    # Brighten the colors
    # Convert to float to prevent clipping during scaling
    output = output.astype(np.float32)
    output = output * 1.2 + 30  # Increase contrast and brightness
    output = np.clip(output, 0, 255).astype(np.uint8)

    explanation = "Simplifying the image by grouping similar colors can help identifying objects as well."
    return output, explanation

def visualize_level_5(frame):
    """
    Combines edge detection and color quantization.
    Highlights edges in red on the color-quantized image.
    """

    # Get the edge-detected image from Level 3
    edge_img, _ = visualize_level_3(frame)  # This is already a color image with red edges
    kernel = np.ones((3, 3), np.uint8)  # 3x3 kernel; adjust size for thickness
    edges_dilated = cv2.dilate(edge_img, kernel, iterations=1)
    # Get the color-quantized image from Level 4
    feature_img, _ = visualize_level_4(frame)  # This is a color image with reduced palette

    # Overlay the edges onto the color-quantized image
    # We'll prioritize the edges by using the edge image where edges are present
    # Create a mask where edges are present (red color)
    edges_gray = cv2.cvtColor(edges_dilated, cv2.COLOR_BGR2GRAY)
    _, edge_mask = cv2.threshold(edges_gray, 50, 255, cv2.THRESH_BINARY)

    # Convert edge_mask to 3 channels
    edge_mask_colored = cv2.cvtColor(edge_mask, cv2.COLOR_GRAY2BGR)

    # Create a red-colored edge image
    red_edges = np.zeros_like(feature_img)
    red_edges[:, :] = [0, 0, 0]  # CHOSE EDGE COLOR HERE

    # Use the edge mask to overlay red edges on the quantized image
    fused_img = np.where(edge_mask_colored == 255, red_edges, feature_img)

    explanation = "All these handcrafted filters can be combined to enhance the detection of objects in the video."
    return fused_img, explanation

### Level 6: Object Detection using Template Matching ### 
def get_color_map(object_names):
    """
    Assigns a unique color to each object name from the COLOR_PALETTE.
    """
    color_map = {}
    for i, name in enumerate(object_names):
        color_map[name] = COLOR_PALETTE[i % len(COLOR_PALETTE)]
    return color_map

def visualize_level_6(frame):
    """
    Detects objects in the frame using template matching.
    Highlights detected objects with colored bounding boxes based on object type.
    Displays the confidence score for each detection.
    """

    # Make a copy of the frame to avoid modifying the original
    frame_copy = frame.copy()

    # Initialize templates on first call
    if not hasattr(visualize_level_6, "templates"):
        visualize_level_6.templates = []
        template_dir = 'templates/'  # Directory where templates are stored

        # Iterate over template files, limit to 5 objects
        for filename in os.listdir(template_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                template_path = os.path.join(template_dir, filename)
                template_gray = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
                if template_gray is not None:
                    # Extract object name from filename (e.g., 'BigPerson.jpg' -> 'Bigperson')
                    object_name = os.path.splitext(filename)[0].replace('_template', '').replace('Template', '').replace('template', '').strip()
                    visualize_level_6.templates.append((object_name, template_gray))
                    if len(visualize_level_6.templates) >= 5:
                        break  # Limit to 5 templates
                else:
                    print(f"Failed to load template: {template_path}")

        if not visualize_level_6.templates:
            print("No valid templates found. Please add template images to the 'templates/' directory.")
        else:
            # Create color map for the loaded objects
            object_names = [obj[0] for obj in visualize_level_6.templates]
            visualize_level_6.color_map = get_color_map(object_names)

    # If no templates loaded, return original frame with explanation
    if not hasattr(visualize_level_6, "templates") or not visualize_level_6.templates:
        explanation = "Step 6: Template images not loaded. Cannot perform object detection."
        return frame_copy, explanation

    # Convert the current frame to grayscale
    gray_frame = cv2.cvtColor(frame_copy, cv2.COLOR_BGR2GRAY)

    # Parameters for template matching
    method = cv2.TM_CCOEFF_NORMED
    threshold = 0.6  # Adjust based on testing

    # Initialize list to hold all detected bounding boxes, confidences, and labels
    detections = []

    # Iterate over each template
    for object_name, template in visualize_level_6.templates:
        template_height, template_width = template.shape

        # Perform template matching
        res = cv2.matchTemplate(gray_frame, template, method)
        loc = np.where(res >= threshold)

        # Collect detections
        for pt in zip(*loc[::-1]):  # Switch columns and rows
            confidence = res[pt[1], pt[0]]
            detections.append({
                'object_name': object_name,
                'position': (int(pt[0]), int(pt[1])),
                'size': (int(template_width), int(template_height)),
                'confidence': float(confidence)
            })

    # Convert detections to NumPy arrays for NMS
    boxes = []
    confidences = []
    labels = []

    for det in detections:
        x, y = det['position']
        w, h = det['size']
        boxes.append([x, y, w, h])
        confidences.append(det['confidence'])
        labels.append(det['object_name'])

    boxes_np = np.array(boxes)

    # Apply Non-Maximum Suppression (NMS)
    final_indices = []
    if len(boxes_np) > 0:
        # Convert to list of tuples for NMS function
        boxes_list = [tuple(box) for box in boxes_np]
        final_indices = non_max_suppression_fast(boxes_np, 0.5)

    # Draw bounding boxes and labels
    for i in final_indices:
        box = boxes_np[i]
        x, y, w, h = box
        object_name = labels[i]
        confidence = confidences[i]

        # Get color for the object
        color = visualize_level_6.color_map.get(object_name, (0, 0, 0))  # Default to black if not found

        # Draw rectangle
        cv2.rectangle(frame_copy, (x, y), (x + w, y + h), color, 2)

        # Prepare label with confidence
        label = f"{object_name}: {confidence:.2f}"

        # Calculate label position
        label_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_ymin = max(y, label_size[1] + 10)
        cv2.rectangle(frame_copy, (x, y - label_size[1] - 10), (x + label_size[0], y), color, cv2.FILLED)
        cv2.putText(frame_copy, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    explanation = "One simple way to detect objects is to compare a template image with each video frame. This is called template matching."
    return frame_copy, explanation

def non_max_suppression_fast(boxes, overlapThresh):
    """
    Applies non-maximum suppression to eliminate redundant overlapping bounding boxes.
    Returns indices of boxes to keep.
    """
    if len(boxes) == 0:
        return []

    # Convert to float if not already
    if boxes.dtype.kind == "i":
        boxes = boxes.astype("float")

    # Initialize the list of picked indexes
    pick = []

    # Grab the coordinates of the bounding boxes
    x1 = boxes[:,0]
    y1 = boxes[:,1]
    x2 = boxes[:,0] + boxes[:,2]
    y2 = boxes[:,1] + boxes[:,3]

    # Compute the area of the bounding boxes and sort by the bottom-right y-coordinate
    area = (x2 - x1 + 1) * (y2 - y1 + 1)
    idxs = np.argsort(y2)

    # Keep looping while some indexes still remain in the indexes list
    while len(idxs) > 0:
        # Grab the last index in the indexes list and add it to the pick list
        last = len(idxs) - 1
        i = idxs[last]
        pick.append(i)

        # Find the largest (x, y) coordinates for the start of the bounding box and the smallest (x, y) coordinates for the end of the bounding box
        xx1 = np.maximum(x1[i], x1[idxs[:last]])
        yy1 = np.maximum(y1[i], y1[idxs[:last]])
        xx2 = np.minimum(x2[i], x2[idxs[:last]])
        yy2 = np.minimum(y2[i], y2[idxs[:last]])

        # Compute the width and height of the bounding box overlap
        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)

        # Compute the ratio of overlap
        overlap = (w * h) / area[idxs[:last]]

        # Delete all indexes from the index list that have overlap greater than the threshold
        idxs = np.delete(idxs, np.concatenate(([last],
            np.where(overlap > overlapThresh)[0])))

    # Return only the bounding boxes that were picked using the integer data type
    return pick

### Using YOLOv8 world ### 
def visualize_level_7(frame, model):
    # Perform object detection
    
    #model.set_classes(["hands", "face", "bottle", "cell phone"])
    results = model(frame, imgsz=1056, verbose=False)[0]  # Get the first result
    
    # Annotate the frame with bounding boxes and labels
    annotated_frame = results.plot()
    
    # Prepare explanation text
    explanation = "Deep learning models use many different filters, or a Neural Network of Convolutions (CNN) to detect objects more accurately."
    
    return annotated_frame, explanation

def visualize_level_8(frame, model_lvl8, track_history):

    results = model_lvl8.track(frame, persist=True, classes=0, imgsz=(640,416), verbose=False)    #67 cell phone   0 person   39: bottle
    
    # Annotate the frame with bounding boxes and labels
    # Get the boxes and track IDsd
    boxes = results[0].boxes.xywh.cpu()
    

    # Visualize the results on the frame
    annotated_frame = results[0].plot()

    # Plot the tracks
    try:
        track_ids = results[0].boxes.id.int().cpu().tolist()
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y)))  # x, y center point
            if len(track) > 120:  
                track.pop(0)

            # Draw the tracking lines
            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
    except Exception as e:
        print(f"Error drawing tracking lines: {e}")

    explanation = "If an object is detected, it can be tracked across frames to follow its movement. Can you draw something?"

    return annotated_frame, explanation

def visualize_level_9(frame, model_lvl9):

    # Perform pose estimation
    results = model_lvl9(frame, imgsz=640, verbose=False)[0]  # Get the first result

    # Visualize the results on the frame
    annotated_frame = results[0].plot()

    explanation = "Essentially, we can track objects like knees and hands relative to eachother to understand human movement and gestures."

    return annotated_frame, explanation



def process_video(cap, model, frame_width, frame_height):
    levels = [
        'Recording',
        'Step 1: The computer sees the world only in numbers of Red, Green and Blue pixel values:',
        'Step 2: Understanding "Convolutions", or how filters work to detect patterns:',
        'Step 3: Detecting edges to find object boundaries:',
        'Step 4: Grouping regions using "fixed palette color quantization":',
        'Step 5: Combining edges and color groups for enhanced detection:',
        'Step 6: Detecting objects using a traditional "template matching" technique:',
        'Step 7: Detecting objects using a deep learning model named You Only Look Once ("YOLO"):',
        'Step 8: Tracking objects in the scene:',
        'Step 9: Analysing human movement:'
    ]

    level_functions = [
        None,  # For recording phase
        visualize_level_1,
        visualize_level_2,
        visualize_level_3,
        visualize_level_4,
        visualize_level_5,
        visualize_level_6,
        visualize_level_7,
        visualize_level_8,
        visualize_level_9
    ]

    index = 0
    total_levels = len(levels)
    level_duration = 40  # seconds
    extended_level_duration = 60 
    frames = []  # To store recorded frames

    while True:
        level_name = levels[index % total_levels]
        level_start_time = time.time()

        # Set level_duration based on level_name
        if level_name == 'Step 6: Detecting objects using a traditional "template matching" technique:' or level_name == 'Step 7: Detecting objects using a deep learning model named You Only Look Once ("YOLO"):' or level_name == 'Step 8: Tracking objects in the scene:':
            level_duration = extended_level_duration
        elif level_name == 'Recording':
            level_duration = 10


        level_name = levels[index % total_levels]
        level_start_time = time.time()

        if level_name == 'Recording':
            # Level 0: Recording Phase
            frames = []
            while time.time() - level_start_time < level_duration:
                ret, frame = cap.read()
                if not ret:
                    print("Failed to grab frame.")
                    break
                # Mirror the frame horizontally
                frame = cv2.flip(frame, 1)
                
                # **Store the original frame without any overlays**
                frames.append(frame.copy())  # Use copy() to ensure stored frames are unmodified
                
                # **Create a separate frame for display with the caption**
                display_frame = frame.copy()
                cv2.putText(display_frame, "Let's record a video to demonstrate computer vision and machine learning.",
                            (10, frame_height - 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
                
                # **Display the frame with the caption**
                cv2.imshow('Visualization - Press q to quit', display_frame)
                
                # Handle key presses
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    return False  # Exit the processing loop
                elif key == ord('a'):
                    index = (index - 1) % total_levels
                    break  # Change level
                elif key == ord('d'):
                    index = (index + 1) % total_levels
                    break  # Change level
            # After recording, proceed to the next level
            index += 1
            continue  # Proceed to next level

        # For other levels, process the recorded frames
        level_func = level_functions[index % total_levels]
        if level_name == 'Step 2: Understanding "Convolutions", or how filters work to detect patterns:':
            # Level 2 code remains the same
            # Copying the Level 2 code from your working implementation

            # Initialize variables for Level 2
            kernel_size = 500  # As per your specification
            block_size = 25    # Adjust as needed
            level_duration = 15  # seconds
            time_per_position = 1  # move kernel once per second
            num_positions = int(level_duration / time_per_position)  # Should be 15

            # Compute kernel positions for 15 steps
            x_positions = np.linspace(0, frame_width - kernel_size, num_positions, dtype=int)
            y_positions = np.linspace(0, frame_height - kernel_size, num_positions, dtype=int)

            # For simplicity, we can move the kernel diagonally from top-left to bottom-right
            kernel_positions = [(x_positions[i], y_positions[i]) for i in range(num_positions)]

            frame_index = 0
            # Initialize a single processed_mask that accumulates over time
            processed_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

            position_index = 0
            position_start_time = time.time()  # Start time for the first kernel position

            while time.time() - level_start_time < level_duration:
                if frame_index >= len(frames):
                    frame_index = 0
                frame = frames[frame_index]
                filtered_frame = enhance_saturation(frame)

                # Check if time to move to next position
                if time.time() - position_start_time >= time_per_position:
                    position_index += 1
                    if position_index >= len(kernel_positions):
                        position_index = 0  # Restart from the beginning
                        # Reset processed_mask when looping back
                        processed_mask.fill(0)
                        print("Reset processed_mask")
                    position_start_time = time.time()  # Reset the start time for the new position

                kernel_position = kernel_positions[position_index]

                # Process the frame using Level 2 function
                output_frame, processed_mask, explanation = visualize_level_2(
                    frame, filtered_frame, processed_mask, kernel_position, kernel_size, block_size)


                # Add explanation text
                overlay = output_frame.copy()
                cv2.rectangle(overlay, (0, frame_height - 70), (frame_width, frame_height), (0, 0, 0), -1)
                alpha = 0.6
                output_frame = cv2.addWeighted(overlay, alpha, output_frame, 1 - alpha, 0)
                # Add level name and explanation
                cv2.putText(output_frame, f"{level_name}", (10, frame_height - 45),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
                if explanation:
                    for i, line in enumerate(explanation.split('\n')):
                        cv2.putText(output_frame, line, (10, frame_height - 15 + i * 15),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
                # Display the processed frame
                cv2.imshow('Visualization - Press q to quit', output_frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    return False  # Exit the processing loop
                elif key == ord('a'):
                    index = (index - 1) % total_levels
                    break  # Change level
                elif key == ord('d'):
                    index = (index + 1) % total_levels
                    break  # Change level

                # Move to next frame
                frame_index += 1
            else:
                index = (index + 1) % total_levels
                continue  # Proceed to next level

        else:
            # Handle Levels 1, 3, 4,  5 , 6 and 7
            while time.time() - level_start_time < level_duration:
                for frame in frames:
            
                    if level_name == 'Step 7: Detecting objects using a deep learning model named You Only Look Once ("YOLO"):':
                        # Apply YOLOv8n to the frame

                        processed_frame, explanation = level_func(frame, model)
                    elif level_name == 'Step 8: Tracking objects in the scene:':
                        # load the model once, if model_lvl8 is none or not defined:
                        if 'model_lvl8' not in locals():
                            model_lvl8 = YOLO("yolov8n.pt")
                        # initialize track_history  once if not defined:
                        if 'track_history' not in locals():
                            track_history = defaultdict(lambda: [])
                        # Process the frame using the appropriate function
                        processed_frame, explanation = level_func(frame, model_lvl8, track_history)
                    
                    elif level_name == 'Step 9: Analysing human movement:':
                        # load the model once, if model_lvl8 is none or not defined:
                        if 'model_lvl9' not in locals():
                            model_lvl9 = YOLO("yolo11n-pose.pt")
                        processed_frame, explanation = level_func(frame, model_lvl9)
                    else:
                        #print("level_name", level_name)
                        # Process the frame using the appropriate function
                        processed_frame, explanation = level_func(frame)
                    
                    # Add explanation text
                    overlay = processed_frame.copy()
                    cv2.rectangle(overlay, (0, frame_height - 70), (frame_width, frame_height), (0, 0, 0), -1)
                    alpha = 0.6
                    processed_frame = cv2.addWeighted(overlay, alpha, processed_frame, 1 - alpha, 0)
                    # Add level name and explanation
                    cv2.putText(processed_frame, f"{level_name}", (10, frame_height - 45),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
                    if explanation:
                        # Handle multi-line explanations
                        for i, line in enumerate(explanation.split('\n')):
                            cv2.putText(processed_frame, line, (10, frame_height - 15 + i * 15),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
                    # Display the processed frame
                    cv2.imshow('Visualization - Press q to quit', processed_frame)
                    
                    
                    
                    
                    
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'):
                        return False  # Exit the processing loop
                    elif key == ord('a'):
                        index = (index - 1) % total_levels
                        break  # Break inner loop to change level
                    elif key == ord('d'):
                        index = (index + 1) % total_levels
                        break  # Break inner loop to change level
                    # Check if time exceeds level duration
                    if time.time() - level_start_time >= level_duration:
                        index = (index + 1) % total_levels
                        break  # Move to next level
                else:
                    continue  # Continue if inner loop wasn't broken
                break  # Break outer loop if level changed

        # Loop through levels
        if index == total_levels:
            index = 0  # Reset index to start from recording again
        elif index == 0:
            continue  # Proceed to recording phase

    return True  # Continue processing

def main():
    model = initialize_model()
    cap, frame_width, frame_height = initialize_camera()
    
    print("Press 'q' during visualization to quit.")
    while True:
        # Process the video through levels, including recording
        continue_processing = process_video(cap, model, frame_width, frame_height)
        if not continue_processing:
            break
    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Application closed.")

if __name__ == "__main__":
    main()


Using GPU (NVIDIA GeForce RTX 4070 Ti) for inference.
Camera 0 opened successfully.
Desired resolution 1920x1200 not supported. Using 1920x1080 instead.
Press 'q' during visualization to quit.
requirements: Ultralytics requirement ['lapx>=0.5.2'] not found, attempting AutoUpdate...
   ---------------------------------------- 1.5/1.5 MB 76.0 MB/s eta 0:00:00

requirements: AutoUpdate success  2.2s, installed 1 package: ['lapx>=0.5.2']
requirements:  Restart runtime or rerun command for updates to take effect

Application closed.
